# Traffy Fondue Data Analysis & Data Quality Pipeline
สมุดงาน (Notebook) นี้ถูกจัดลำดับขั้นตอนการประมวลผลข้อมูลอย่างเป็นระบบ แบ่งออกเป็น 5 ส่วนหลัก:
1. **Data Ingestion** - ดึงข้อมูลจาก Traffy Fondue Public API
2. **Data Overview & Exploration** - สำรวจมิติข้อมูล โครงสร้าง คอลัมน์ และค่าที่สูญหาย
3. **Data Cleaning & Transformation** - ทำความสะอาดข้อมูล แปลงประเภท Datetime แยกพิกัด และคำนวณเวลาใหม่
4. **Data Quality & Business Logic Validation** - ตรวจสอบความถูกต้อง ความสมเหตุสมผลของลำดับเวลา และกรณีดึงเคสกลับมาแก้ไข (Internal Rework)
5. **Final Output** - ตรวจสอบชุดข้อมูลที่ผ่านการทำความสะอาดและพร้อมใช้งาน

## 0. Data Ingestion
ดึงข้อมูลสถิติ Traffy Fondue ประจำเดือนจาก Public API เข้ามาเป็น Pandas DataFrame

In [1]:
import requests
from io import StringIO
import pandas as pd 
import numpy as np

# กำหนด URL และ Parameters สำหรับดึงข้อมูล
url = "https://publicapi.traffy.in.th/teamchadchart-stat-api/download/bangkok_monthly"

params = {
    "name": "Thanatip Nitinantakul",
    "org": "KMUTT",
    "email": "champthanatip2005@gmail.com",
    "purpose": "For practice and make personal project",
    "tel": "0955391162",
    "file_name": "bangkok_2026-07"
}

response = requests.get(url, params=params)
print("Status Code:", response.status_code)

if response.status_code == 200:
    df = pd.read_csv(StringIO(response.text))
    print("Data loaded successfully!")
else:
    print("Failed to load data.")

Status Code: 200
Data loaded successfully!


## 1. Data Overview & Exploration
สำรวจภาพรวมของข้อมูลเบื้องต้น เช่น จำนวนแถว-คอลัมน์, ประเภทข้อมูล, ค่าว่าง (Missing Values), ข้อมูลซ้ำซ้อน (Duplicates) และค่าในคอลัมน์สำคัญ

In [2]:
print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")
print("\n--- Column Overview ---")
df.info()

Number of rows: 37,135
Number of columns: 24

--- Column Overview ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37135 entries, 0 to 37134
Data columns (total 24 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ticket_id                    37135 non-null  object 
 1   type                         37135 non-null  object 
 2   organization                 37135 non-null  object 
 3   organization_action          37135 non-null  object 
 4   comment                      36968 non-null  object 
 5   coords                       37135 non-null  object 
 6   photo                        37135 non-null  object 
 7   photo_after                  23756 non-null  object 
 8   address                      36968 non-null  object 
 9   subdistrict                  37135 non-null  object 
 10  district                     37135 non-null  object 
 11  province                     37135 non-null  object 
 12  time

In [3]:
# ตรวจสอบ Missing Values ในแต่ละคอลัมน์
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print("--- Missing Values by Column ---")
print(missing)

# ตรวจสอบแถวที่ซ้ำซ้อน (Duplicate Rows)
print("\n--- Duplicate Check ---")
print(f"Are there duplicates?: {df.duplicated().any()} ({df.duplicated().sum()} rows)")

--- Missing Values by Column ---
star                           28333
timestamp_finished             14610
duration_minutes_finished      14610
photo_after                    13379
timestamp_inprogress            7147
duration_minutes_inprogress     7147
duration_minutes_total          5335
comment                          167
address                          167
dtype: int64

--- Duplicate Check ---
Are there duplicates?: False (0 rows)


In [4]:
pd.set_option('display.max_columns', None)  # แสดงคอลัมน์ทั้งหมด
print("--- Sample 5 Rows ---")
display(df.sample(5))

print("\n--- Unique States ---")
print(df['state'].unique())

--- Sample 5 Rows ---


,ticket_id,type,organization,organization_action,comment,coords,photo,photo_after,address,subdistrict,district,province,timestamp,state,star,count_reopen,last_activity,duration_minutes_inprogress,duration_minutes_finished,duration_minutes_total,timestamp_inprogress,timestamp_finished,message_id,problemtype_tag
9992,2026-VXKUYE,ต้นไม้และสวนสาธารณะ -> ขอให้ตัดแต่งต้นไม้ กิ่งไม้,"ร้องทุกข์ กทม. 1555, กรุงเทพมหานคร, เขตวัฒนา, ...","ฝ่ายรักษาความสะอาดฯ เขตวัฒนา, เขตวัฒนา, กรุงเท...",ปัญหา: ริมถนนดังกล่าว บริเวณหน้าบ้านเลขที่ 110...,"100.58003,13.72338",https://storage.googleapis.com/traffy_public_b...,https://storage.googleapis.com/traffy_public_b...,แขวงคลองตันเหนือ เขตวัฒนา กรุงเทพมหานคร,คลองตันเหนือ,วัฒนา,กรุงเทพมหานคร,2026-07-08 21:25:58.893518,เสร็จสิ้น,NaN,0,2026-07-09 13:18:29.907552,773.0,953.0,953.0,2026-07-09 10:18:28.748857,2026-07-09 13:18:29.907552,2063845,"{""ต้นไม้และสวนสาธารณะ -> ขอให้ตัดแต่งต้นไม้ กิ..."
27868,2026-8RGHPP,ไฟฟ้า,"กรุงเทพมหานคร, เขตบางซื่อ, สำนักการโยธา กทม., ...","สำนักการโยธา กทม., เขตบางซื่อ, ศูนย์เครื่องมือ...",แจ้ง ไฟส่องฟุตบาท ดับยาวนาน หลายเดือนแล้ว \n\n...,"100.52702,13.82616",https://storage.googleapis.com/traffy_public_b...,NaN,9/61 ถ. วงศ์สว่าง แขวงวงศ์สว่าง บางซื่อ กรุงเท...,วงศ์สว่าง,บางซื่อ,กรุงเทพมหานคร,2026-07-22 20:18:43.112807,กำลังดำเนินการ,NaN,0,2026-07-23 10:23:32.746701,845.0,NaN,845.0,2026-07-23 10:23:13.143798,NaN,2102333,{ไฟฟ้า}
33500,2026-WJRUR6,ผิดกฎจราจร,"กรุงเทพมหานคร, เขตธนบุรี, สน.บางยี่เรือ, กองบั...","เขตธนบุรี, กองบังคับการตำรวจนครบาล 8 (บก.น.8),...",แจ้งหลายทีแล้วสำหรับรถคันนี้ ไม่เห็นมีการดำเนิ...,"100.48128,13.72167",https://storage.googleapis.com/traffy_public_b...,NaN,แขวงบางยี่เรือ เขตธนบุรี กรุงเทพมหานคร,บางยี่เรือ,ธนบุรี,กรุงเทพมหานคร,2026-07-27 21:44:23.946568,กำลังดำเนินการ,NaN,0,2026-07-28 07:19:04.267827,NaN,NaN,NaN,NaN,NaN,2110323,{ผิดกฎจราจร}
8249,2026-DD2CC6,ไฟฟ้า,"กรุงเทพมหานคร, เขตดอนเมือง, การไฟฟ้านครหลวง ME...","บริษัท โทรคมนาคมแห่งชาติ จำกัด (มหาชน), การไฟฟ...",พบสายไฟบริเวณสะพานหน้าโครงการพาร์ควิว วิภาวดี ...,"100.58427,13.89142",https://storage.googleapis.com/traffy_public_b...,https://storage.googleapis.com/traffy_public_b...,VHRM+HM8 แขวงดอนเมือง ดอนเมือง กรุงเทพมหานคร 1...,ดอนเมือง,ดอนเมือง,กรุงเทพมหานคร,2026-07-07 16:07:32.67694,กำลังดำเนินการ,NaN,0,2026-08-08 14:58:19.33344,3136.0,28553.0,28553.0,2026-07-09 20:23:18.710484,2026-07-27 12:00:30.5238,2061249,{ไฟฟ้า}
11554,2026-YP337J,หาบเร่แผงลอย,"กรุงเทพมหานคร, เขตพระนคร, ฝ่ายเทศกิจ เขตพระนคร","ฝ่ายเทศกิจ เขตพระนคร, เขตพระนคร, กรุงเทพมหานคร",ร้านอาหารตั้งโต๊ะสีเหลืองเต็มหน้า guest house ...,"100.49855,13.75819",https://storage.googleapis.com/traffy_public_b...,https://storage.googleapis.com/traffy_public_b...,QF5X+7CG 216 ถ. ข้าวสาร แขวงตลาดยอด เขตพระนคร ...,ตลาดยอด,พระนคร,กรุงเทพมหานคร,2026-07-09 21:16:14.226861,เสร็จสิ้น,NaN,0,2026-07-15 22:03:39.428324,730.0,8688.0,8688.0,2026-07-10 09:25:59.630445,2026-07-15 22:03:39.428324,2066002,{หาบเร่แผงลอย}



--- Unique States ---
['เสร็จสิ้น' 'กำลังดำเนินการ' 'รอรับเรื่อง']


In [5]:
# --- Check Unique Values for Categorical Columns ---
categorical_cols = ['state', 'type', 'district', 'subdistrict', 'province']

print("=== Summary of Unique Counts in Categorical Columns ===")
for col in categorical_cols:
    print(f"{col}: {df[col].nunique():,} unique values")

print("\n--- Unique Values in 'state' ---")
print(df['state'].value_counts())

print("\n--- Top 10 Problem Types in 'type' ---")
print(df['type'].value_counts().head(10))

print("\n--- Unique Values in 'province' ---")
print(df['province'].unique())

=== Summary of Unique Counts in Categorical Columns ===
state: 3 unique values
type: 187 unique values
district: 51 unique values
subdistrict: 172 unique values
province: 2 unique values

--- Unique Values in 'state' ---
state
เสร็จสิ้น         21729
กำลังดำเนินการ    15365
รอรับเรื่อง          41
Name: count, dtype: int64

--- Top 10 Problem Types in 'type' ---
type
ผิดกฎจราจร           4133
ถนน                  3871
ไฟฟ้า                2823
ความสะอาด            2797
ทางเท้า              2683
อุปกรณ์ชำรุด         2390
ต้นไม้               1655
เสียง                1256
อาคารสถานที่ชำรุด    1243
หาบเร่แผงลอย          986
Name: count, dtype: int64

--- Unique Values in 'province' ---
['กรุงเทพมหานคร' 'นนทบุรี']


## 2. Data Cleaning & Transformation Pipeline
ขั้นตอนการทำความสะอาดและแปลงโครงสร้างข้อมูลทีละขั้นตอน (Step-by-Step Data Pipeline)

### Step 1: Dynamic Validation & Handling of `problemtype_tag` (with Text Normalization)
ใช้ฟังก์ชัน **Text Normalization / Canonicalization** เพื่อปรับรูปแบบข้อความของ `problemtype_tag` และ `type` ให้เป็นมาตรฐานเดียวกันก่อนเปรียบเทียบ:
1. ลบเครื่องหมายครอบ: `{ } [ ] ' "`
2. ปรับการเว้นวรรคหลังเครื่องหมายจุลภาค (Comma Spacing) ให้สม่ำเสมอ
3. ตัดช่องว่างด้านหน้า-หลัง และยุบช่องว่างซ้ำซ้อน

**Business Logic:**
* หากเหมือนกัน 100% $\rightarrow$ ทำการ Drop ทิ้งได้อย่างปลอดภัย (ลด Data Redundancy)
* หากยังมีข้อมูลที่ต่างกันจริง (เช่น มี multi-tags เพิ่มเติม) $\rightarrow$ เก็บไว้และเปลี่ยนชื่อเป็น `tags` (ป้องกัน Silent Data Loss)

In [6]:
import re

def normalize_tag(text):
    """
    ฟังก์ชันปรับข้อความให้อยู่ในรูปมาตรฐาน (Canonical Format):
    1. ลบเครื่องหมายครอบ: { } [ ] ' "
    2. จัดรูปแบบเครื่องหมาย Comma ให้เว้นวรรคสม่ำเสมอ
    3. ตัดช่องว่างส่วนเกินด้านหน้า-หลัง และยุบช่องว่างซ้ำ
    """
    if pd.isna(text):
        return ""
    cleaned = re.sub(r"[\{\}\[\]\'\"]", "", str(text))
    cleaned = re.sub(r"\s*,\s*", ", ", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned

df_clean = df.copy()

# Dynamic Validation: เปรียบเทียบหลังผ่านการ Normalize
if 'problemtype_tag' in df_clean.columns:
    norm_tag = df_clean['problemtype_tag'].apply(normalize_tag)
    norm_type = df_clean['type'].apply(normalize_tag)
    
    is_identical = (norm_tag == norm_type).all()
    
    if is_identical:
        df_clean.drop(columns=['problemtype_tag'], inplace=True)
        print("Info: 'problemtype_tag' matches 'type' 100% (after normalization) -> Dropped safely.")
    else:
        diff_count = (norm_tag != norm_type).sum()
        df_clean.rename(columns={'problemtype_tag': 'tags'}, inplace=True)
        print(f"Warning: 'problemtype_tag' has {diff_count:,} rows with distinct values -> Preserved and renamed to 'tags'.")

Info: 'problemtype_tag' matches 'type' 100% (after normalization) -> Dropped safely.


### Step 2: Parse Problem Type Hierarchy (`main_category`, `sub_category`, `detail_category`)
คอลัมน์ `type` มีโครงสร้างแบบ **Hierarchical Taxonomy** (คั่นด้วย `->`) เช่น `"สาธารณูปโภค -> โทรศัพท์ -> ขอให้จัดการสายสื่อสาร..."`
ทำการแตกหมวดหมู่ออกเป็นระดับชั้นแบบ **Dynamic Hybrid** เพื่อความยืดหยุ่นและรองรับการทำ Dashboard Drill-down:
1. `main_category`: หมวดหมู่หลัก (Level 1)
2. `sub_category`: หมวดหมู่ย่อย (Level 2)
3. `detail_category`: รายละเอียดประเภทปัญหา (Level 3) *(และสร้างระดับถัดไปอัตโนมัติหากพบในอนาคต)*

In [7]:
# 1. แตกคอลัมน์ type ออกตาม '->' แบบ Dynamic
type_split = df_clean['type'].fillna('').str.split('->', expand=True)

# 2. ตั้งชื่อคอลัมน์ตามระดับความลึกที่ตรวจพบจริง
level_names = ['main_category', 'sub_category', 'detail_category']
col_names = [
    level_names[i] if i < len(level_names) else f'sub_category_{i}'
    for i in range(type_split.shape[1])
]
type_split.columns = col_names

# 3. Clean ช่องว่างและเติม None ในค่าว่าง
for col in type_split.columns:
    type_split[col] = type_split[col].str.strip().replace('', 'None').fillna('None')

# รวมเข้ากับตารางหลัก
df_clean = pd.concat([df_clean, type_split], axis=1)

print(f"Successfully parsed 'type' hierarchy into {list(type_split.columns)}.")
display(df_clean[['type', 'main_category', 'sub_category', 'detail_category']].head(5))

Successfully parsed 'type' hierarchy into ['main_category', 'sub_category', 'detail_category'].


,type,main_category,sub_category,detail_category
0,ไฟฟ้า,ไฟฟ้า,None,None
1,ไฟฟ้า,ไฟฟ้า,None,None
2,เหตุเดือดร้อนรำคาญ -> กลิ่น,เหตุเดือดร้อนรำคาญ,กลิ่น,None
3,กระทำผิดในที่สาธารณะ -> ขอให้ดำเนินการกับผู้ตั...,กระทำผิดในที่สาธารณะ,ขอให้ดำเนินการกับผู้ตั้งวางสิ่งของกีดขวาง,None
4,ความสะอาด,ความสะอาด,None,None


### Step 3: Fill Missing Values
สำหรับคอลัมน์ข้อความที่มีค่าว่าง เช่น `comment` และ `address` ให้เติมด้วย **"Not specified"** เพื่อป้องกันปัญหาในการประมวลผลข้อความ

In [8]:
df_clean['comment'] = df_clean['comment'].fillna("Not specified")
df_clean['address'] = df_clean['address'].fillna("Not specified")

print("Current missing values in 'comment':", df_clean['comment'].isna().sum())
print("Current missing values in 'address':", df_clean['address'].isna().sum())

Current missing values in 'comment': 0
Current missing values in 'address': 0


### Step 4: Convert Datetime Data Types
แปลงคอลัมน์วันที่และเวลาจากประเภท String ให้เป็น **Datetime Objects** เพื่อให้สามารถคำนวณและเปรียบเทียบช่วงเวลาได้

In [9]:
date_columns = ['timestamp', 'last_activity', 'timestamp_inprogress', 'timestamp_finished']

for col in date_columns:
    df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')

print("Successfully converted time columns. Current dtypes:")
print(df_clean[date_columns].dtypes)

Successfully converted time columns. Current dtypes:
timestamp               datetime64[ns]
last_activity           datetime64[ns]
timestamp_inprogress    datetime64[ns]
timestamp_finished      datetime64[ns]
dtype: object


### Step 5: Split Coordinates
แยกคอลัมน์พิกัด `coords` (lon,lat) ออกเป็น 2 คอลัมน์เดี่ยว: `longitude` และ `latitude` เพื่อความสะดวกในการวิเคราะห์เชิงพื้นที่ (GIS/Map Plotting)

In [10]:
df_clean[['longitude', 'latitude']] = df_clean['coords'].str.split(',', expand=True).astype(float)
df_clean.drop(columns=['coords'], inplace=True)

display(df_clean[['longitude', 'latitude']].head(3))

,longitude,latitude
0,100.50134,13.69287
1,100.56445,13.77374
2,100.62973,13.69652


### Step 6: Extract Organization Summary Features
เนื่องจากคอลัมน์ `organization` และ `organization_action` มีหลายหน่วยงานต่อหนึ่งเรื่อง เพื่อป้องกันปัญหา **Sparse Data (คอลัมน์ขยะ NaN)** และ **Schema Drift** ในตารางหลัก จึงทำการสกัดเฉพาะ **Summary Features** ที่สำคัญสำหรับการวิเคราะห์:
1. `primary_org`: หน่วยงานแรกที่รับเรื่อง (ดึงจาก `organization`)
2. `latest_action_org`: หน่วยงานล่าสุดที่ลงมือจัดการ/แก้ไขจริง (ดึงจาก `organization_action`)
3. `org_count`: จำนวนหน่วยงานทั้งหมดที่เกี่ยวข้องในเคสนี้

In [11]:
# 1. ดึงหน่วยงานแรกที่รับเรื่อง (Primary Org)
df_clean['primary_org'] = df_clean['organization'].fillna('').apply(
    lambda x: x.split(',')[0].strip() if x else 'Not specified'
)

# 2. ดึงหน่วยงานล่าสุดที่ลงมือจัดการ (Latest Action Org)
df_clean['latest_action_org'] = df_clean['organization_action'].fillna('').apply(
    lambda x: x.split(',')[0].strip() if x else 'Not specified'
)

# 3. นับจำนวนหน่วยงานที่เกี่ยวข้องทั้งหมด
df_clean['org_count'] = df_clean['organization'].fillna('').apply(
    lambda x: len(x.split(',')) if x else 0
)

print("Successfully extracted organization summary features (primary_org, latest_action_org, org_count).")
display(df_clean[['ticket_id', 'primary_org', 'latest_action_org', 'org_count']].head(5))

Successfully extracted organization summary features (primary_org, latest_action_org, org_count).


,ticket_id,primary_org,latest_action_org,org_count
0,2026-73DL6J,กรุงเทพมหานคร,Bangkok Smart Lighting (สำนักการโยธา กทม.),6
1,2026-UCLP6R,กรุงเทพมหานคร,ฝ่ายโยธา เขตดินแดง,7
2,2026-GA9YRR,ร้องทุกข์ กทม. 1555,ฝ่ายเทศกิจ เขตพระโขนง,5
3,2026-ARKV8W,ร้องทุกข์ กทม. 1555,ฝ่ายรักษาความสะอาดฯ เขตพระโขนง,5
4,2026-2U67VX,กรุงเทพมหานคร,ฝ่ายสิ่งแวดล้อมฯ เขตประเวศ,4


### Step 7: Recalculate & Standardize Durations
คำนวณระยะเวลาจริงแบบละเอียด (Absolute Lead Time):
1. `calculated_from_start`: ระยะเวลาทั้งหมดตั้งแต่ส่งเรื่องจนเสร็จสิ้น (`timestamp_finished - timestamp`)
2. `calculated_from_inprogress`: ระยะเวลาปฏิบัติงานจริงของเจ้าหน้าที่ (`timestamp_finished - timestamp_inprogress`)
3. อัปเดต `duration_minutes_total` ด้วยค่าที่คำนวณใหม่หากเคสเสร็จสิ้นแล้ว

In [12]:
# 1. คำนวณเวลารวมตั้งแต่รับเรื่องจนเสร็จ (นาที)
df_clean['calculated_from_start'] = (df_clean['timestamp_finished'] - df_clean['timestamp']).dt.total_seconds() / 60

# 2. คำนวณเวลาทำงานของเจ้าหน้าที่ (นาที)
df_clean['calculated_from_inprogress'] = (df_clean['timestamp_finished'] - df_clean['timestamp_inprogress']).dt.total_seconds() / 60

# 3. อัปเดต duration_minutes_total สำหรับเคสที่เสร็จสิ้น
df_clean['duration_minutes_total'] = np.where(
    df_clean['timestamp_finished'].notna(), 
    df_clean['calculated_from_start'], 
    df_clean['duration_minutes_total']
)

print("Successfully calculated and updated durations.")

Successfully calculated and updated durations.


### Step 8: Feature Engineering (Internal Rework Flag)
สร้างตัวแปร `is_internal_rework` เพื่อระบุเคสที่เคยมีการบันทึกเวลาเสร็จสิ้นรอบแรกแล้ว แต่สถานะยังไม่ใช่ 'เสร็จสิ้น' และไม่ได้เกิดจากการกด Reopen ของประชาชน (เคสที่ถูกดึงกลับมาแก้ไขภายใน)

In [13]:
# สร้าง Feature Flag ระบุเคสที่ส่งต่อ/แก้ไขภายใน
df_clean['is_internal_rework'] = (
    df_clean['timestamp_finished'].notna() & 
    (df_clean['state'] != 'เสร็จสิ้น') & 
    (df_clean['count_reopen'] == 0)
)

print(f"จำนวนเคสที่เป็น Internal Rework: {df_clean['is_internal_rework'].sum():,} เคส")

จำนวนเคสที่เป็น Internal Rework: 215 เคส


## 3. Data Quality & Business Logic Validation
ส่วนการตรวจสอบคุณภาพและความสมเหตุสมผลของข้อมูล (Data Validation / Sanity Checks)

### Validation 1: เปรียบเทียบระยะเวลาจากระบบ vs ระยะเวลาที่คำนวณจริงจาก Timestamp
ตรวจสอบว่าระยะเวลาที่ระบบบันทึกมา ตรงกับผลต่างของ Timestamp จริงหรือไม่

In [14]:
comparison_cols = [
    'duration_minutes_total',       # เวลา total ที่คำนวณจาก Timestamp แจ้ง
    'calculated_from_start',        # เวลาคำนวณเอง (จากตอนแจ้ง)
    'duration_minutes_finished',    # เวลา finished เดิมของระบบ
    'calculated_from_inprogress'    # เวลาคำนวณเอง (จากตอนรับเรื่อง)
]

print("--- ตัวอย่างเปรียบเทียบระยะเวลา (เฉพาะเคสที่ทำงานเสร็จแล้ว 10 แถวแรก) ---")
display(df_clean[df_clean['timestamp_finished'].notna()][comparison_cols].head(10).round(2))

--- ตัวอย่างเปรียบเทียบระยะเวลา (เฉพาะเคสที่ทำงานเสร็จแล้ว 10 แถวแรก) ---


,duration_minutes_total,calculated_from_start,duration_minutes_finished,calculated_from_inprogress
0,2949.05,2949.05,2950.0,1771.20
1,12998.99,12998.99,12999.0,12179.66
2,9190.41,9190.41,9191.0,8699.80
3,7725.76,7725.76,7726.0,7234.95
5,2470.30,2470.30,2471.0,2165.20
6,52626.96,52626.96,52627.0,52218.06
7,5558.48,5558.48,5559.0,5053.30
8,72825.93,72825.93,72826.0,72419.87
9,48460.67,48460.67,48461.0,48056.51
11,1582.24,1582.24,1583.0,418.42


### Validation 2: ตรวจสอบเคสที่ยังไม่เสร็จสิ้น แต่มีเวลาเสร็จสิ้นบันทึกอยู่
ตรวจสอบความสัมพันธ์ระหว่าง `state`, `timestamp_finished` และ `count_reopen` เพื่อวิเคราะห์เคสที่มีการทำงานหลายรอบ

In [15]:
# เคสที่ยังไม่เสร็จสิ้น แต่มี timestamp_finished บันทึกไว้
unfinished_with_finish_time = df_clean['timestamp_finished'].notna() & (df_clean['state'] != 'เสร็จสิ้น')

print(f"จำนวนเคสที่ยังไม่เสร็จ แต่มีเวลาเสร็จสิ้น: {unfinished_with_finish_time.sum():,} แถว")
print(f"  - เป็นการ Reopen โดยประชาชน (count_reopen > 0): {(unfinished_with_finish_time & (df_clean['count_reopen'] > 0)).sum():,} แถว")
print(f"  - เป็นการแก้ไข/ส่งต่อภายใน (is_internal_rework): {df_clean['is_internal_rework'].sum():,} แถว")

print("\n--- ตัวอย่างเคส Internal Rework 5 แถวแรก ---")
display(df_clean[df_clean['is_internal_rework']][['ticket_id', 'type', 'state', 'count_reopen', 'timestamp', 'timestamp_finished', 'last_activity']].head(5))

จำนวนเคสที่ยังไม่เสร็จ แต่มีเวลาเสร็จสิ้น: 796 แถว
  - เป็นการ Reopen โดยประชาชน (count_reopen > 0): 581 แถว
  - เป็นการแก้ไข/ส่งต่อภายใน (is_internal_rework): 215 แถว

--- ตัวอย่างเคส Internal Rework 5 แถวแรก ---


,ticket_id,type,state,count_reopen,timestamp,timestamp_finished,last_activity
71,2026-AMEXHH,อาคารสถานที่ชำรุด,กำลังดำเนินการ,0,2026-07-01 05:47:43.355706,2026-08-05 14:41:22.572868,2026-08-25 13:03:04.349559
165,2026-N78AKP,ไฟฟ้า,กำลังดำเนินการ,0,2026-07-01 07:43:04.584865,2026-07-08 09:55:32.637963,2026-07-08 10:48:40.954614
190,2026-DADWGW,สาธารณูปโภค -> โทรศัพท์ -> ขอให้ดำเนินการสายโท...,กำลังดำเนินการ,0,2026-07-01 07:57:47.236397,2026-07-10 09:14:30.338666,2026-07-10 09:14:53.912248
247,2026-VNWUWY,ต้นไม้,กำลังดำเนินการ,0,2026-07-01 08:27:17.484590,2026-07-28 12:56:11.684273,2026-07-31 15:50:11.259634
670,2026-N38UT4,ผิดกฎจราจร,กำลังดำเนินการ,0,2026-07-01 14:00:17.755784,2026-07-02 09:54:36.707131,2026-07-06 13:35:50.878287


### Validation 3: ตรวจสอบลำดับเวลา (Chronological Consistency)
ตรวจสอบว่า `last_activity` เกิดขึ้นถูกต้องตามตรรกะเวลาหรือไม่ (ต้องเท่ากับหรือหลัง `timestamp_finished` เสมอ ไม่มีทางเกิดขึ้นก่อนหน้า)

In [16]:
# 1. กรองเฉพาะเคสที่มีเวลาเสร็จสิ้น
has_finished = df_clean['timestamp_finished'].notna()

# 2. ตรวจสอบความผิดปกติ: last_activity เกิดขึ้นก่อน timestamp_finished
invalid_activity = has_finished & (df_clean['last_activity'] < df_clean['timestamp_finished'])

print(f"จำนวนเคสทั้งหมดที่มีเวลาเสร็จสิ้น: {has_finished.sum():,} แถว")
print(f"จำนวนเคสที่ผิดปกติ (last_activity < timestamp_finished): {invalid_activity.sum():,} แถว")

if invalid_activity.sum() > 0:
    print("\nพบข้อมูลที่ลำดับเวลาผิดปกติ:")
    cols_to_show = ['ticket_id', 'state', 'count_reopen', 'timestamp', 'timestamp_finished', 'last_activity']
    display(df_clean[invalid_activity][cols_to_show].head(10))
else:
    print("ลำดับเวลาถูกต้องสมบูรณ์: ไม่มีเคสใดที่ last_activity เกิดก่อนเวลาเสร็จสิ้น")

# สรุปพฤติกรรมเวลาเพิ่มเติม
df_finished = df_clean[has_finished]
exact_same = (df_finished['last_activity'] == df_finished['timestamp_finished']).sum()
after_finished = (df_finished['last_activity'] > df_finished['timestamp_finished']).sum()

print("\n--- พฤติกรรมความเคลื่อนไหวล่าสุด (last_activity vs timestamp_finished) ---")
print(f"1. เวลาตรงกันพอดี (ปิดงานแล้วจบเลย): {exact_same:,} แถว ({exact_same/len(df_finished)*100:.2f}%)")
print(f"2. เกิดขึ้นหลังจากนั้น (มีการอัปเดต/ส่งต่อ/ทำต่อภายหลัง): {after_finished:,} แถว ({after_finished/len(df_finished)*100:.2f}%)")

จำนวนเคสทั้งหมดที่มีเวลาเสร็จสิ้น: 22,525 แถว
จำนวนเคสที่ผิดปกติ (last_activity < timestamp_finished): 0 แถว
ลำดับเวลาถูกต้องสมบูรณ์: ไม่มีเคสใดที่ last_activity เกิดก่อนเวลาเสร็จสิ้น

--- พฤติกรรมความเคลื่อนไหวล่าสุด (last_activity vs timestamp_finished) ---
1. เวลาตรงกันพอดี (ปิดงานแล้วจบเลย): 20,067 แถว (89.09%)
2. เกิดขึ้นหลังจากนั้น (มีการอัปเดต/ส่งต่อ/ทำต่อภายหลัง): 2,458 แถว (10.91%)


## 4. Final Cleaned Output
ตรวจสอบภาพรวมของชุดข้อมูล `df_clean` ที่ผ่านขั้นตอนการทำความสะอาดและตรวจสอบคุณภาพครบถ้วนแล้ว

In [17]:
print(f"Final Dataset Shape: {df_clean.shape[0]:,} rows, {df_clean.shape[1]} columns")
print("\n--- Final 5 Sample Rows ---")
display(df_clean.head())

Final Dataset Shape: 37,135 rows, 33 columns

--- Final 5 Sample Rows ---


,ticket_id,type,organization,organization_action,comment,photo,photo_after,address,subdistrict,district,province,timestamp,state,star,count_reopen,last_activity,duration_minutes_inprogress,duration_minutes_finished,duration_minutes_total,timestamp_inprogress,timestamp_finished,message_id,main_category,sub_category,detail_category,longitude,latitude,primary_org,latest_action_org,org_count,calculated_from_start,calculated_from_inprogress,is_internal_rework
0,2026-73DL6J,ไฟฟ้า,"กรุงเทพมหานคร, เขตบางคอแหลม, Bangkok Smart Lig...","Bangkok Smart Lighting (สำนักการโยธา กทม.), สำ...",ไปเกาะกลางตรงร้านบ้านกรองน้ำดับ 1 ดวง,https://storage.googleapis.com/traffy_public_b...,https://storage.googleapis.com/traffy_public_b...,140 ถ. พระรามที่ 3 แขวงบางคอแหลม เขตบางคอแหลม ...,บางคอแหลม,บางคอแหลม,กรุงเทพมหานคร,2026-07-01 00:00:33.353971,เสร็จสิ้น,NaN,0,2026-07-03 01:09:36.175143,1178.0,2950.0,2949.047020,2026-07-01 19:38:24.305371,2026-07-03 01:09:36.175143,2049056,ไฟฟ้า,None,None,100.50134,13.69287,กรุงเทพมหานคร,Bangkok Smart Lighting (สำนักการโยธา กทม.),6,2949.047020,1771.197830,False
1,2026-UCLP6R,ไฟฟ้า,"กรุงเทพมหานคร, เขตดินแดง, ฝ่ายโยธา เขตดินแดง, ...","ฝ่ายโยธา เขตดินแดง, Bangkok Smart Lighting (สำ...",มีเสาไฟฟ้าดับประมาณ 3 ต้นคะ ตรงซอยประชาสงเคราะ...,https://storage.googleapis.com/traffy_public_b...,https://storage.googleapis.com/traffy_public_b...,251 ซอย ประชาสงเคราะห์ 24 แขวงดินแดง เขตดินแดง...,ดินแดง,ดินแดง,กรุงเทพมหานคร,2026-07-01 00:02:29.543952,เสร็จสิ้น,5.0,0,2026-08-02 21:40:41.197138,820.0,12999.0,12998.994957,2026-07-01 13:41:49.698335,2026-07-10 00:41:29.241357,2049057,ไฟฟ้า,None,None,100.56445,13.77374,กรุงเทพมหานคร,ฝ่ายโยธา เขตดินแดง,7,12998.994957,12179.659050,False
2,2026-GA9YRR,เหตุเดือดร้อนรำคาญ -> กลิ่น,"ร้องทุกข์ กทม. 1555, กรุงเทพมหานคร, เขตพระโขนง...","ฝ่ายเทศกิจ เขตพระโขนง, เขตพระโขนง, ฝ่ายสิ่งแวด...",ปัญหา: ภายในซอยดังกล่าว พบมีการนำขยะหลายประเภท...,https://storage.googleapis.com/traffy_public_b...,https://storage.googleapis.com/traffy_public_b...,แขวงบางจาก เขตพระโขนง กรุงเทพมหานคร,บางจาก,พระโขนง,กรุงเทพมหานคร,2026-07-01 00:03:10.013345,เสร็จสิ้น,NaN,0,2026-07-07 09:13:34.624667,491.0,9191.0,9190.410189,2026-07-01 08:13:46.847721,2026-07-07 09:13:34.624667,2049058,เหตุเดือดร้อนรำคาญ,กลิ่น,None,100.62973,13.69652,ร้องทุกข์ กทม. 1555,ฝ่ายเทศกิจ เขตพระโขนง,5,9190.410189,8699.796282,False
3,2026-ARKV8W,กระทำผิดในที่สาธารณะ -> ขอให้ดำเนินการกับผู้ตั...,"ร้องทุกข์ กทม. 1555, กรุงเทพมหานคร, เขตพระโขนง...","ฝ่ายรักษาความสะอาดฯ เขตพระโขนง, ฝ่ายเทศกิจ เขต...",ปัญหา: ภายในซอยดังกล่าว พบมีการนำขยะหลายประเภท...,https://storage.googleapis.com/traffy_public_b...,https://storage.googleapis.com/traffy_public_b...,แขวงบางจาก เขตพระโขนง กรุงเทพมหานคร,บางจาก,พระโขนง,กรุงเทพมหานคร,2026-07-01 00:03:48.440694,เสร็จสิ้น,NaN,0,2026-07-06 08:49:33.980161,491.0,7726.0,7725.758991,2026-07-01 08:14:36.938355,2026-07-06 08:49:33.980161,2049059,กระทำผิดในที่สาธารณะ,ขอให้ดำเนินการกับผู้ตั้งวางสิ่งของกีดขวาง,None,100.62973,13.69652,ร้องทุกข์ กทม. 1555,ฝ่ายรักษาความสะอาดฯ เขตพระโขนง,5,7725.758991,7234.950697,False
4,2026-2U67VX,ความสะอาด,"กรุงเทพมหานคร, เขตประเวศ, ฝ่ายเทศกิจ เขตประเวศ...","ฝ่ายสิ่งแวดล้อมฯ เขตประเวศ, ฝ่ายเทศกิจ เขตประเ...",ที่ว่างตรงเสาไฟฟ้าแรงสูง ตรงบึง3ของซอยพัฒนาการ...,https://storage.googleapis.com/traffy_public_b...,NaN,แขวงประเวศ เขตประเวศ กรุงเทพมหานคร,ประเวศ,ประเวศ,กรุงเทพมหานคร,2026-07-01 00:04:55.505176,กำลังดำเนินการ,NaN,0,2026-08-05 00:24:32.060688,412.0,NaN,412.000000,2026-07-01 06:56:47.177991,NaT,2049060,ความสะอาด,None,None,100.67237,13.72937,กรุงเทพมหานคร,ฝ่ายสิ่งแวดล้อมฯ เขตประเวศ,4,NaN,NaN,False
